# Deep learning on images

## Load modules from repo

In [1]:
# Le code suivant dans un notebook permet de :
# - autoriser les imports de fichiers python de ce repo
# - spécifier les chemins relativement à la racine du repo plutôt que relativement au notebook

import os

# Ce code cherche le dossier racine en remontant dans l'arborescence
# jusqu'à ce qu'il trouve le dossier 'src'.
# Cela le rend indépendant de l'endroit où vous lancez le notebook.
try:
    # On part du dossier du notebook
    notebook_dir = os.path.dirname(__file__)
except NameError:
    # __file__ n'existe pas en mode interactif, on utilise le répertoire de travail
    notebook_dir = os.getcwd()

# On remonte jusqu'à trouver un dossier contenant 'src'
project_root = notebook_dir
while not os.path.isdir(os.path.join(project_root, 'src')):
    parent_dir = os.path.dirname(project_root)
    if parent_dir == project_root: # On a atteint la racine du système
        raise FileNotFoundError("Impossible de trouver le dossier 'src'. Vérifiez la structure du projet.")
    project_root = parent_dir

os.chdir(project_root)

In [2]:
os.getcwd()

'/home/val/Documents/Dev/DataScientest/Rakuten'

In [ ]:
import src
from src.preprocessing.core import load_reproducible_split
from src.preprocessing.image import get_image_path, load_image, get_image_features_with_hash, get_image_md5_hash
from src.preprocessing.pipelines.deep_learning_on_images import preprocess_features, load_preprocessors, save_preprocessors
from models.on_images.deep_learning import define_model

2025-09-30 16:39:09.437142: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-30 16:39:09.910740: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-30 16:39:11.262189: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [4]:
import importlib
importlib.reload(src.preprocessing.core)
importlib.reload(src.preprocessing.image)
importlib.reload(src.preprocessing.pipelines.deep_learning_on_images)

<module 'src.preprocessing.pipelines.deep_learning_on_images' from '/home/val/Documents/Dev/DataScientest/Rakuten/src/preprocessing/pipelines/deep_learning_on_images.py'>

## Load tensorflow

In [5]:
import tensorflow as tf

In [6]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [7]:
print(f"tensorflow: {tf.__version__}")

tensorflow: 2.20.0


## Load split dataset

In [8]:
X_train, X_test, y_train, y_test = load_reproducible_split(folder = 'Dataset2')

## Preprocessing

In [9]:
from pathlib import Path
artifacts_folder=Path('artifacts/on_images/deep_learning/v1')

# fit_preprocessors=True
fit_preprocessors=False

small_train_sample = True  # TODO: set to False for real training or fitting preprocessors

In [10]:
if fit_preprocessors and small_train_sample:
    raise ValueError("When fit_preprocessors=True, small_train_sample should be False so that encoders are fitted on the full training dataset.")

In [11]:
if small_train_sample:
    print(f"{small_train_sample=}")
    X_train=X_train.sample(frac=.1)
    y_train=y_train.loc[X_train.index]

small_train_sample=True


In [12]:
print(X_train.shape)

(6793, 31)


In [13]:
import joblib
if fit_preprocessors:
    preprocessors = {}
    print('will fit')
else: # load preprocessors
    print('loading')
    preprocessors = load_preprocessors(names=['target','tabular','hash'],artifacts_folder=artifacts_folder)

loading
artifacts/on_images/deep_learning/v1/preprocessors/target.joblib
artifacts/on_images/deep_learning/v1/preprocessors/tabular.joblib
artifacts/on_images/deep_learning/v1/preprocessors/hash.joblib


In [14]:
train_ds, new_preprocessors, train_inputs_dict = preprocess_features(X_train, y_train, preprocessors, BATCH_SIZE = 32)
preprocessors = preprocessors | new_preprocessors
n_cols_tabular = train_inputs_dict['tabular_input'].shape[1]
n_cols_tabular

I0000 00:00:1759243153.482396    5410 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4143 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


24

In [15]:
new_preprocessors

{}

In [16]:
# Save on disk
if fit_preprocessors:
    print('Saving.')
    save_preprocessors(new_preprocessors,artifacts_folder)

In [17]:
test_ds, new_preprocessors, test_inputs_dict = preprocess_features(X_test, y_test, preprocessors, BATCH_SIZE = 32)

## Model architecture

In [ ]:
raise Exception("You can skip to running section 'Loading model'.")

In [18]:
from tensorflow import keras
from tensorflow.keras import layers

### Inputs

In [19]:
image_input = keras.Input(shape=(500, 500, 3), name="image_input")
tabular_input = keras.Input(shape=(n_cols_tabular,), name='tabular_input')

In [20]:
pHash_vocab_size = len(preprocessors['hash'].categories_[0])
md5_vocab_size = len(preprocessors['hash'].categories_[1])
embedding_dim = 16  # La taille souhaitée du vecteur pour chaque hash. C'est un hyperparamètre.
pHash_vocab_size, md5_vocab_size

(62446, 63866)

In [21]:
pHash_input = keras.Input(shape=(1,), name='pHash_input', dtype='int64')  # Embedding a besoin du type int
md5_input = keras.Input(shape=(1,), name='md5_input', dtype='int64')

In [ ]:
dense_layers_sizes=[]

### Image branch

In [ ]:
# On utilise un modèle pré-entraîné.
# C'est une base très puissante pour traiter les images.
base_model = keras.applications.EfficientNetV2B0(
    include_top=False, # On ne garde que les couches d'extraction de features
    weights='imagenet', # Poids appris sur des millions d'images
    input_tensor=image_input
)
base_model.trainable = False # On "gèle" le modèle de base pour le début de l'entraînement

# On ajoute nos propres couches par-dessus
image_features = layers.GlobalAveragePooling2D(name='image_pooling')(base_model.output)
dense_layers_sizes.append(128)
image_features = layers.Dense(dense_layers_sizes[-1], activation='relu', name='image_dense')(image_features)

### Tabular branch

In [ ]:
dense_layers_sizes.append(64)
tabular_features = layers.Dense(dense_layers_sizes[-1], activation='relu', name='tabular_dense_1')(tabular_input)
dense_layers_sizes.append(32)
tabular_features = layers.Dense(dense_layers_sizes[-1], activation='relu', name='tabular_dense_2')(tabular_features)

### Hash branches

Chaque hash passe par sa propre couche d'Embedding.

In [24]:
pHash_features = layers.Embedding(input_dim=pHash_vocab_size, output_dim=embedding_dim, name='pHash_embedding')(pHash_input)
pHash_features = layers.Flatten(name='pHash_flatten')(pHash_features)  # retire une dimension superflue de taille 1

md5_features = layers.Embedding(input_dim=md5_vocab_size, output_dim=embedding_dim, name='md5_embedding')(md5_input)
md5_features = layers.Flatten(name='md5_flatten')(md5_features)


### Fusing branches


In [25]:
# On fusionne toutes les features apprises en un seul grand vecteur
all_features = layers.concatenate([
    image_features,
    tabular_features,
    pHash_features,
    md5_features
])

### Classification

In [ ]:
# Quelques couches denses pour apprendre les interactions entre les différentes modalités
dense_layers_sizes.append(256)
x = layers.Dense(dense_layers_sizes[-1], activation='relu', name='final_dense_1')(all_features)
x = layers.Dropout(0.5)(x)  # pour éviter l'overfitting
num_classes = 27  # nombre de classes
output = layers.Dense(num_classes, activation='softmax', name='output')(x)

### Model

In [ ]:
model = keras.Model(
    inputs=[image_input, tabular_input, pHash_input, md5_input],
    outputs=output
)

### New (TODO)

In [ ]:
#TODO: remove old
model = define_model(embedding_dim=16, n_cols_tabular=n_cols_tabular, num_classes=27)

### Load a saved model

In [ ]:
#    Pour charger un modèle sauvegardé on utilise la fonction load_model() de tensorflow.keras.models.
model_loaded = keras.models.load_model(artifacts_folder / 'best_model-1.keras')

### Model summary

In [ ]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 500, 500,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 500, 500,  │          0 │ image_input[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 500, 500,  │          0 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 250, 250,  │        864 │ normalization[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 250, 250,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 250, 250,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 250, 250,  │      4,608 │ stem_activation[… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 250, 250,  │         64 │ block1a_project_… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, 250, 250,  │          0 │ block1a_project_… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_conv │ (None, 125, 125,  │      9,216 │ block1a_project_… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_bn   │ (None, 125, 125,  │        256 │ block2a_expand_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_act… │ (None, 125, 125,  │          0 │ block2a_expand_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_project_co… │ (None, 125, 125,  │      2,048 │ block2a_expand_a… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_project_bn  │ (None, 125, 125,  │        128 │ block2a_project_… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_conv │ (None, 125, 125,  │     36,864 │ block2a_project_… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_bn   │ (None, 125, 125,  │        512 │ block2b_expand_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_act… │ (None, 125, 125,  │          0 │ block2b_expand_b

 Total params: 12,654,275 (48.27 MB)

 Trainable params: 2,244,987 (8.56 MB)

 Non-trainable params: 5,919,312 (22.58 MB)

 Optimizer params: 4,489,976 (17.13 MB)

## Training

### Saving a model

In [53]:
# Pick an available filename to save a model.
k=1
while True:
    new_location_for_saving_model = Path(artifacts_folder / f'best_model-{k}.keras')
    if not new_location_for_saving_model.exists():
        break
    k+=1
new_location_for_saving_model_weights = Path(artifacts_folder / f'best_model-{k}.h5')
new_location_for_saving_model, new_location_for_saving_model_weights


(PosixPath('artifacts/on_images/deep_learning/v1/best_model-2.keras'),
 PosixPath('artifacts/on_images/deep_learning/v1/best_model-2.h5'))

In [ ]:
# # Callback pour sauvegarder le meilleur modèle au fur et à mesure
save = ModelCheckpoint(
    new_location_for_saving_model_weights,
    save_best_only=True,
    save_weights_only=True,
    monitor='val_accuracy',
    mode='max'
)

In [ ]:
# # ou sauvegarder l'intégralité du modèle après l'entraînement, c'est-à-dire son architecture et ses poids :
# model.save(new_location_for_saving_model)

### Training

In [38]:
model.optimizer.learning_rate

<Variable path=adam/learning_rate, shape=(), dtype=float32, value=0.0010000000474974513>

In [33]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [34]:
# callbacks = [save]
callbacks = []

In [ ]:
raise Exception("Are you sure you want to launch training?")

In [35]:
model_history = model.fit(train_ds, validation_data=test_ds, epochs=2, callbacks=callbacks)

Epoch 1/2


2025-09-30 16:46:53.921208: I external/local_xla/xla/service/service.cc:163] XLA service 0x70c3a4002890 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-09-30 16:46:53.921261: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9
2025-09-30 16:46:54.273560: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-09-30 16:46:55.820471: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300
2025-09-30 16:46:56.851699: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-09-30 16:46:56.

212/213 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - accuracy: 0.2742 - loss: 2.5719

2025-09-30 16:47:53.037479: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-09-30 16:47:53.131945: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-09-30 16:47:53.705628: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-09-30 16:47:53.805287: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-09-30 16:47:54.427563: E external/local_xla/xla/stream_

213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 200ms/step - accuracy: 0.2747 - loss: 2.5702

2025-09-30 16:49:12.203358: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-09-30 16:49:19.919077: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-09-30 16:49:20.017448: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-09-30 16:49:20.824995: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please invest

213/213 ━━━━━━━━━━━━━━━━━━━━ 159s 596ms/step - accuracy: 0.3694 - loss: 2.2063 - val_accuracy: 0.4941 - val_loss: 1.7053
Epoch 2/2
213/213 ━━━━━━━━━━━━━━━━━━━━ 89s 418ms/step - accuracy: 0.5073 - loss: 1.6425 - val_accuracy: 0.5381 - val_loss: 1.5457


## Evaluation